In [5]:
import requests
from bs4 import BeautifulSoup
import json
from time import sleep

In [16]:
def find_facts_heading(soup):
    # Try different possible heading variations
    possible_headings = [
        'Interesting Facts',
        'Fun Facts',
        'Facts',
        'INTERESTING FACTS',
        'FUN FACTS'
    ]
    
    # Try different heading tags (h1, h2, h3, etc.)
    for tag in ['h1', 'h2', 'h3', 'h4']:
        for heading_text in possible_headings:
            # Try exact match
            heading = soup.find(tag, string=heading_text)
            if heading:
                return heading
                
            # Try partial match
            heading = soup.find(tag, string=lambda text: text and heading_text.lower() in text.lower())
            if heading:
                return heading
    
    return None

In [38]:
try:
        # Add headers to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        # Make the request to the website
        print(f"Fetching content from {'https://www.potterswaxmuseum.com/celebrities/harry-potter/'}...")
        response = requests.get('https://www.potterswaxmuseum.com/celebrities/harry-potter/', headers=headers)
        response.raise_for_status()
        
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Print the full HTML for debugging (commented out by default)
        # print(soup.prettify())
        
        # Find the facts heading
        facts_heading = find_facts_heading(soup)
        
        if not facts_heading:
            print("\nDebug information:")
            print("Could not find facts heading. Available headings on the page:")
            for tag in ['h1', 'h2', 'h3', 'h4']:
                headings = soup.find_all(tag)
                for heading in headings:
                    print(f"{tag}: {heading.get_text().strip()}")
            
        print(f"Found heading: {facts_heading.get_text().strip()}")
        
        # Get all elements after the heading
        current_element = facts_heading.find_next_sibling()
        
        # Get sting in Current element
        current_element_text = current_element.get_text().strip()
        # print(current_element_text)

        ## Split the text into a list using newline as delimiter
        facts_list = current_element_text.split('\n')

        # Print the list with indices for clarity
        for i, fact in enumerate(facts_list, 1):
            print(f"{i}. {fact}")
            
except requests.RequestException as e:
        print(f"Error fetching the webpage: {e}")
except Exception as e:
        print(f"An error occurred: {e}")


Fetching content from https://www.potterswaxmuseum.com/celebrities/harry-potter/...
Found heading: Interesting Facts
1. The houses of Harry Potter’s school, Hogwarts, were invented on an airplane sick bag.
2. The four houses of Hogwarts School of Witchcraft and Wizardry — Ravenclaw, Slytherin, Hufflepuff, and Gryffindor — are nearly as iconic as Harry Potter himself. However, the history of their names is not as serious as the history of Harry Potter’s name. In a tweet on December 15, 2017, J. K. Rowling said, “The best thing I ever wrote on was an airplane sick bag. Came up with the Hogwarts houses on it.”
3. Stephen Spielberg almost directed the first Harry Potter film.
4. Steven Spielberg is one of Hollywood’s most beloved directors; from Jaws to Jurassic Park to Schindler’s List, Spielberg has touched almost every major film and film franchise in Hollywood today — and Harry Potter is no exception. Spielberg was attached to the film almost from the beginning and had even picked out 

In [42]:
facts_head = []
facts_content = []

for i in range(0, len(facts_list), 2):
    facts_head.append(facts_list[i])
    facts_content.append(facts_list[i+1])

In [43]:
facts_head

['The houses of Harry Potter’s school, Hogwarts, were invented on an airplane sick bag.',
 'Stephen Spielberg almost directed the first Harry Potter film.',
 'Some parts of Harry Potter are based on real life.',
 'On May 2nd, the world celebrates Harry Potter’s own personal holiday, International Harry Potter Day.',
 'The actor who played Harry Potter’s best friend rapped in his audition for the movie.',
 'J. K. Rowling and Harry Potter share birth dates.',
 'J. K. Rowling insisted that the actors in the Harry Potter movies be British and/or Irish.',
 'The Harry Potter spells are in Latin.',
 'Harry Potter actor Daniel Radcliffe got the role due to a well-timed coincidence.']

In [44]:
facts_content

['The four houses of Hogwarts School of Witchcraft and Wizardry — Ravenclaw, Slytherin, Hufflepuff, and Gryffindor — are nearly as iconic as Harry Potter himself. However, the history of their names is not as serious as the history of Harry Potter’s name. In a tweet on December 15, 2017, J. K. Rowling said, “The best thing I ever wrote on was an airplane sick bag. Came up with the Hogwarts houses on it.”',
 'Steven Spielberg is one of Hollywood’s most beloved directors; from Jaws to Jurassic Park to Schindler’s List, Spielberg has touched almost every major film and film franchise in Hollywood today — and Harry Potter is no exception. Spielberg was attached to the film almost from the beginning and had even picked out Haley Joel Osment (from Sixth Sense fame) to play Harry Potter. However, eventually, Spielberg decided to leave the project to spend more time with his family.',
 'Although many Harry Potter fans are still waiting patiently for their Hogwarts letter to arrive, most have s

In [53]:
# Create a dictionary from the two lists using a dictionary comprehension with id as key, head as value, and content as value
facts_dict = {i: {'head': head, 'content': content} for i, head, content in zip(range(1, len(facts_head)+1), facts_head, facts_content)}

In [55]:
# Save the dictionary to a JSON file
with open('harry_potter_facts.json', 'w') as file:
    json.dump(facts_dict, file, indent=4)
    